## Product Category Translation and Analysis

In [ ]:
# import required libraries
import numpy as np
import pandas as pd

In [ ]:
date_cols = [
    "order_purchase_timestamp",
     "order_approved_at",
     "order_delivered_carrier_date",
     "order_delivered_customer_date",
     "order_estimated_delivery_date",
     "review_creation_date",
     "review_answer_timestamp"
]
supply_chain = pd.read_csv("../data/processed/supply_chain_autopsy.csv", parse_dates=date_cols)


In [ ]:
# Load olist_products_dataset
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
products.sample(5)

In [ ]:
# Load product_category_name_translation dataset
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")
category_translation.head()

In [ ]:
# Merge tables
merged_df = supply_chain.merge(products, on="product_id").merge(category_translation, on="product_category_name")
merged_df.sample()

In [ ]:
english_product_metrics = merged_df.groupby("product_category_name_english").agg(
    total_orders = ("order_id", "nunique"),
    total_delayed_orders = ('delivery_delay_days', lambda x: (x > 0).sum()),
    avg_freight_value = ("freight_value", "mean")
).reset_index()

In [ ]:
english_product_metrics["avg_freight_value"] = english_product_metrics["avg_freight_value"].round(2)

# Failure rate by state
english_product_metrics["failure_rate_pct"] = (
    (english_product_metrics["total_delayed_orders"] / english_product_metrics["total_orders"]) * 100).round(2)

In [ ]:
orders_gt_100 = english_product_metrics[english_product_metrics["total_orders"] > 100]

In [ ]:
orders_gt_100.sort_values("failure_rate_pct", ascending=False).head(10).reset_index(drop=True)

##### Observation

The "audio" product category has the highest delivery failure rate (12.07%) among categories with over 100 orders, while "health_beauty" and "bed_bath_table" have among the lowest failure rates (around 8.3-8.4%), suggesting that product type influences logistics reliability.

### Extra

##### Removing the products_category_name and keeping only prdocut_category_name_english

In [ ]:
english_col = merged_df.pop('product_category_name_english')

idx = merged_df.columns.get_loc('product_category_name')

# Remove the old and insert the new at that specific spot
merged_df.drop(columns=['product_category_name'], inplace=True)
merged_df.insert(idx, 'product_category_name_english', english_col)
# Rename the column to product_category_name
merged_df = merged_df.rename(columns={"product_category_name_english": "product_category_name"})

In [ ]:
# merged_df.to_csv("../data/processed/catgory_name_translated.csv", index=False)